# GPT-v1 — Full Research Notebook (decoder-only, GPT-1 paper style)

Architecture: **Radford et al. 2018 GPT-1** — decoder-only Transformer, causal masked
self-attention, learned token + position embeddings, GELU, LayerNorm post-LN blocks,
next-token language modeling objective. The architecture code lives in
**`transformer.py`** (source of truth) — this notebook adds the full research stack:

| Feature | Where |
|---|---|
| Single config cell (paper vs local size) | Cell 3 |
| Data: DailyDialog + Alpaca + GSM8K, word-level vocab | Part 1 |
| EDA: repeated lines, examples, length histograms (altair) | Part 2 |
| Checkpoint rollback: `last` / `final` / `step_*` / `epoch_*` + RNG restore | Part 3 |
| Resume training from latest saved point (no restart) | Part 4 |
| Token accuracy, perplexity, quality dashboard | Part 5 |
| Zero-shot generation samples | Part 6 |
| Full pipeline (train → eval → charts → dashboard → samples) | Part 7 |
| Architecture summary + parameter table (torchinfo) | Part 8 |
| **Deep model visualization**: torchviz computation graph incl. backward, TensorBoard real-time weight/gradient histograms, netron ONNX | Part 9 |
| Gradio chat UI (Gradio 6 compatible) | Part 10 |

> **Real-time backpropagation watch:** Part 4 training writes scalars + weight/gradient
> histograms to `runs/`. While training runs, open a terminal and start
> `tensorboard --logdir=runs` for live, colorful gradient/weight curves.


In [1]:
# Cell 1 — Setup + import transformer.py (architecture source of truth)
import os, math, random, time, copy, glob, re
from os.path import exists
import torch
import torch.nn as nn

os.environ["RUN_MODE"] = "local"   # local | full | kaggle
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0), "| vram MB:", torch.cuda.get_device_properties(0).total_memory // (1024 * 1024))

import importlib.util
from pathlib import Path
mod_path = Path("transformer.py").resolve()
spec = importlib.util.spec_from_file_location("gptv1", mod_path)
gptv1 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gptv1)
print("Loaded:", mod_path)
print("Paper GPT-1 reference:", gptv1.PAPER_GPT1)


torch: 2.11.0+cu128 | cuda: True
gpu: NVIDIA GeForce GTX 1650 | vram MB: 3724


/home/creator/miniconda3/envs/transformer-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


No HF_TOKEN set; anonymous HF downloads (rate limits possible).
Loaded: /home/creator/Desktop/Deeplearning/own_trainied_models/GPT_models_version/gpt-v1/transformer.py
Paper GPT-1 reference: {'n_layer': 12, 'n_embd': 768, 'n_head': 12, 'd_ff': 3072, 'max_seq_len': 512, 'base_lr': 0.00025, 'warmup': 2000, 'dropout': 0.1, 'weight_decay': 0.01}


In [2]:
# Cell 2 — SINGLE CONFIG CELL (edit everything here)
CONFIG_OVERRIDE = {
    # ----- schedule -----
    "num_epochs": 5,
    "batch_size": 8,
    "accum_iter": 8,
    "base_lr": 2.5e-4,      # paper max LR
    "warmup": 2000,         # paper warmup steps
    "weight_decay": 0.01,   # paper
    "grad_clip": 1.0,
    "save_every_steps": 2000,

    # ----- architecture (local-safe GPT-1 family, same structure as paper) -----
    # Paper: n_layer=12, n_embd=768, n_head=12, d_ff=3072, max_seq_len=512
    "n_layer": 8,
    "n_embd": 512,
    "n_head": 8,
    "d_ff": 2048,           # 4 * n_embd
    "dropout": 0.1,
    "max_seq_len": 128,
    "block_stride": 96,

    # ----- HF datasets -----
    "train_sources": ["dailydialog", "alpaca", "gsm8k"],
    "valid_sources": ["dailydialog", "gsm8k"],
    "alpaca_train_pairs": 52000,
    "gsm8k_train_pairs": 7000,
    "gsm8k_valid_pairs": 1000,

    # ----- files -----
    "vocab_path": "gptv1_vocab_local.json",
    "file_prefix": "gptv1_local_",

    # ----- research extras (notebook-level) -----
    "tensorboard": True,        # live weight/gradient histograms while training
    "tb_dir": "runs/gptv1_local",
}
print("CONFIG_OVERRIDE ready. Keys:", sorted(CONFIG_OVERRIDE.keys()))


CONFIG_OVERRIDE ready. Keys: ['accum_iter', 'alpaca_train_pairs', 'base_lr', 'batch_size', 'block_stride', 'd_ff', 'dropout', 'file_prefix', 'grad_clip', 'gsm8k_train_pairs', 'gsm8k_valid_pairs', 'max_seq_len', 'n_embd', 'n_head', 'n_layer', 'num_epochs', 'save_every_steps', 'tb_dir', 'tensorboard', 'train_sources', 'valid_sources', 'vocab_path', 'warmup', 'weight_decay']


In [3]:
# Cell 3 — preview merged config + architecture banner
cfg = gptv1.get_run_config(CONFIG_OVERRIDE)
print("Merged config:")
for k in sorted(cfg.keys()):
    print(f"  {k}: {cfg[k]}")
print()
gptv1.print_architecture(cfg)


Merged config:
  accum_iter: 8
  alpaca_train_pairs: 52000
  base_lr: 0.00025
  batch_size: 8
  betas: (0.9, 0.999)
  block_stride: 96
  d_ff: 2048
  data_seed: 42
  dropout: 0.1
  file_prefix: gptv1_local_
  grad_clip: 1.0
  gsm8k_train_pairs: 7000
  gsm8k_valid_pairs: 1000
  max_seq_len: 128
  mode: local
  n_embd: 512
  n_head: 8
  n_layer: 8
  num_epochs: 5
  save_every_steps: 2000
  tb_dir: runs/gptv1_local
  tensorboard: True
  train_pairs: None
  train_sources: ['dailydialog', 'alpaca', 'gsm8k']
  valid_pairs: None
  valid_sources: ['dailydialog', 'gsm8k']
  vocab_path: gptv1_vocab_local.json
  warmup: 2000
  weight_decay: 0.01

GPT-v1 architecture (paper-inspired)
Paper GPT-1: L=12, d=768, heads=12, d_ff=3072, ctx=512
This run: L=8, d=512, heads=8, d_ff=2048, ctx=128


# Part 1: Data & Vocabulary

DailyDialog (chat) + Alpaca (instructions) + GSM8K (math reasoning), formatted as
`User: ...\nAssistant: ...` turns, then encoded into contiguous LM blocks
(context `max_seq_len`, stride `block_stride`, paper style) for next-token training.

**Rollback point:** the vocab file `vocab_path` is saved after first build, so on fresh
kernel start this step re-loads it from disk instead of rebuilding.


In [4]:
# Build texts, vocabulary, LM blocks
train_texts, valid_texts = gptv1.build_train_valid_texts(cfg)
print(f"Train texts: {len(train_texts)} | Valid texts: {len(valid_texts)}")

vocab = gptv1.build_vocab(train_texts, cfg["vocab_path"])
print(f"Vocabulary: {vocab.get_vocab_size()} tokens | <pad>={vocab.token_to_id('<pad>')} "
      f"<bos>={vocab.token_to_id('<bos>')} <eos>={vocab.token_to_id('<eos>')} <unk>={vocab.token_to_id('<unk>')}")

train_loader, valid_loader = gptv1.create_dataloaders_for_lm(vocab, train_texts, valid_texts, cfg)
print(f"Train LM blocks: {len(train_loader)} batches | Valid LM blocks: {len(valid_loader)} batches")


Data sources -> train: dailydialog,alpaca,gsm8k | valid: dailydialog,gsm8k
Train texts: 135026 | Valid texts: 8069
Loaded vocab gptv1_vocab_local.json size=30000
Vocabulary: 30000 tokens | <pad>=0 <bos>=1 <eos>=2 <unk>=3
Train LM blocks: 7882 batches | Valid LM blocks: 389 batches


# Part 2: Visualization & EDA

View 1: which training/validation lines repeat. View 2: example prompts, token-length
histograms, and source-distribution counts — colorful altair charts.


In [5]:
# Visualization state + helpers
import collections
VIZ_STATE = {"did_train": False, "history": [], "epoch_rows": [], "val_loss": None}

def print_step(num, total, title, note=""):
    print(f"\n========================================================================\n"
          f"STEP {num}/{total}: {title}\n{note}\n"
          f"========================================================================\n", flush=True)


In [6]:
# EDA — repeated lines, examples, length histograms
def eda_repeated(limit=6):
    print("View 1: repeated lines in train/valid")
    for name, texts in [("train", train_texts), ("valid", valid_texts)]:
        cnt = collections.Counter(texts)
        reps = [(t, n) for t, n in cnt.items() if n > 1]
        reps.sort(key=lambda x: -x[1])
        print(f"-- {name}: {len(texts)} texts, {len(reps)} repeated lines, "
              f"{sum(n for _, n in reps):,} duplicate occurrences")
        for t, n in reps[:limit]:
            print(f"   x{n:3d}  {t[:80]}")
        if not reps:
            print("   (no repeats)")
    return reps

eda_repeated()


View 1: repeated lines in train/valid
-- train: 135026 texts, 6004 repeated lines, 12,765 duplicate occurrences
   x  7  User: Thank you very much.
Assistant: You're welcome.
   x  6  User: Excuse me.
Assistant: Yes?
   x  6  User: Here you are.
Assistant: Thank you.
   x  6  User: Thank you.
Assistant: You're welcome.
   x  5  User: But she means a lot to me.
Assistant: Then forgive her mistake.
   x  5  User: Thanks a lot.
Assistant: You are welcome.
-- valid: 8069 texts, 71 repeated lines, 144 duplicate occurrences
   x  4  User: Excuse me.
Assistant: Yes?
   x  2  User: It's about a ten-minute walk.
Assistant: That's very clear. I think I can 
   x  2  User: Yes, very urgent. And there is no other fax machine around. And the office
   x  2  User: Looks good! I ’ ll squeeze us some fresh orange juice.
Assistant: Get the 
   x  2  User: Would you please mail these letters, Lucy?
Assistant: Yes, Sir.
   x  2  User: Well, suffer then. If you have a toothache, you have to go to the dent

[('User: Excuse me.\nAssistant: Yes?', 4),
 ("User: It's about a ten-minute walk.\nAssistant: That's very clear. I think I can find my way now. Thank you.",
  2),
 ('User: Yes, very urgent. And there is no other fax machine around. And the office of our boss is closed, so I can not use his machine either. And his secretary is out. Do you have any suggestion?\nAssistant: Either call this extension number 0085 or ask them to send the fax to you as e-mail, you print it out.',
  2),
 ('User: Looks good! I ’ ll squeeze us some fresh orange juice.\nAssistant: Get the jam and butter while you are in there! Oh! And don ’ t forget the syrup!',
  2),
 ('User: Would you please mail these letters, Lucy?\nAssistant: Yes, Sir.', 2),
 ('User: Well, suffer then. If you have a toothache, you have to go to the dentist.\nAssistant: It always hurts. I hate going.',
  2),
 ('User: Oh!!! I have a horrible toothache.\nAssistant: You should go to the dentist.',
  2),
 ('User: Yeah. For five minutes, everythin

In [7]:
# EDA view 2: examples + token-length histograms (altair)
import pandas as pd
import altair as alt
alt.data_transformers.disable_max_rows()   # allow charts > 5000 rows

def eda_lengths():
    rows = []
    for name, texts in [("train", train_texts[:20000]), ("valid", valid_texts)]:
        for t in texts:
            rows.append({"split": name, "len": len(vocab.encode(t).ids)})
    df = pd.DataFrame(rows)
    print(f"Token-length stats: train mean {df[df.split=='train'].len.mean():.1f} | "
          f"valid mean {df[df.split=='valid'].len.mean():.1f} | max {df.len.max()}")
    chart = alt.Chart(df).mark_bar(opacity=0.75).encode(
        alt.X("len:Q", bin=alt.Bin(maxbins=40), title="tokens per line"),
        alt.Y("count()", title="count"),
        alt.Color("split:N", title="split"),
    ).properties(title="Token-length histogram (train vs valid)", width=520, height=250)
    return chart

print("Example prompts (first 3 train / 2 valid):")
for split, texts in [("train", train_texts), ("valid", valid_texts)]:
    for t in texts[: 3 if split == "train" else 2]:
        print(f"[{split}] {t[:100]}")

_eda_len_chart = eda_lengths()
display(_eda_len_chart)


Example prompts (first 3 train / 2 valid):
[train] User: Solve this step by step. Gerald thought it would be funny to trick his coworker, who's allergi
[train] User: Thank you. How much is the fare?
Assistant: The meter reads 9.15 dollars.
[train] User: Explain the historical significance of the Cuban Missile Crisis.
Assistant: The Cuban Missile 
[valid] User: Which direction should I take?
Assistant: Make a left on this next street.
[valid] User: I think I caught a cold.
Assistant: Do you have any symptoms?
Token-length stats: train mean 42.7 | valid mean 35.0 | max 551


alt.Chart(...)

# Part 3: Checkpointing — rollback to the latest saved point

`last.pt` = latest state (every save, mid-epoch too) · `final.pt` = finished model ·
`step_N.pt` / `epoch_N.pt` = historical snapshots (old ones auto-cleaned, keep 3).
Checkpoints store model + optimizer + cosine LR state + **RNG state** (CPU + CUDA), so a
killed kernel resumes deterministically from the exact saved step.


In [8]:
# Checkpoint helpers (notebook level, wrapping transformer.py)
def checkpoint_paths(config):
    return gptv1.checkpoint_paths(config)

def collect_rng_state():
    return gptv1.collect_rng_state()

def restore_rng_state(state):
    return gptv1.restore_rng_state(state)

def load_ckpt(path, map_location="cpu"):
    return torch.load(path, map_location=map_location)

def unwrap_model_state(obj):
    if isinstance(obj, dict) and "model" in obj and "optimizer" in obj:
        return obj["model"], obj
    return obj, None

def build_checkpoint(module, optimizer, epoch, step, config, val_loss, history=None, epoch_rows=None):
    # gptv1 payload + research extras (history for charts)
    payload = gptv1.build_checkpoint(module, optimizer, None, epoch, step, config, val_loss)
    payload["viz_history"] = list(history or [])
    payload["epoch_rows"] = list(epoch_rows or [])
    return payload

def save_checkpoint(path, payload):
    torch.save(payload, path)
    print(f"Saved {path}", flush=True)

def cleanup_old_checkpoints(prefix, keep=3):
    gptv1.cleanup_old_checkpoints(prefix, keep=keep)

print("Checkpoint helpers ready.")


Checkpoint helpers ready.


# Part 4: Training worker — resume from last point + real-time TensorBoard

- **Rollback:** if `last.pt` exists → model/optimizer/LR/RNG restored, training
  continues from the exact saved step (no restart from zero).
- **Real-time backpropagation:** with `"tensorboard": True`, every optimizer step a
  `SummaryWriter` logs loss/LR scalars, and periodically **weight + gradient
  histograms** for every parameter (colorful curves — watch biases and weights move
  live). Run `tensorboard --logdir=runs` in a terminal while this cell runs.
- Epochs still capped by `num_epochs` in the config cell; `final.pt` written at the end
  makes future runs load-only (zero training).


In [9]:
# Teacher-forced next-token accuracy (LM, ignore <pad>)
def token_accuracy_lm(model, valid_loader, pad_id, max_batches=40):
    device = next(model.parameters()).device
    model.eval()
    correct = total = 0
    for i, (xb, yb) in enumerate(valid_loader):
        xb, yb = xb.to(device), yb.to(device)
        logits, _ = model(xb)
        pred = logits.argmax(-1)
        m = yb != pad_id
        correct += int((pred[m] == yb[m]).sum().item())
        total += int(m.sum().item())
        if i + 1 >= max_batches:
            break
    return correct / max(total, 1), correct, total


In [10]:
# Full training worker (resume + history + TensorBoard)
def train_worker(tokenizer, train_loader, valid_loader, config):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Train worker using device: {device}", flush=True)
    print(f"RUN_MODE={config['mode']} L={config['n_layer']} d={config['n_embd']} "
          f"epochs={config['num_epochs']}", flush=True)

    model = gptv1.make_model(tokenizer.get_vocab_size(), config).to(device)
    module = model
    if torch.cuda.device_count() > 1:
        print(f"Using DataParallel across {torch.cuda.device_count()} GPUs", flush=True)
        model = nn.DataParallel(model)
        module = model.module

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=float(config["base_lr"]),
        betas=tuple(config.get("betas", (0.9, 0.999))),
        weight_decay=float(config.get("weight_decay", 0.01)),
    )

    paths = checkpoint_paths(config)
    save_every = int(config.get("save_every_steps", 1200))
    start_epoch, global_step = 0, 0
    history, epoch_rows = [], []
    last_val_loss = None

    if exists(paths["last"]):
        print(f"Resuming from {paths['last']}", flush=True)
        ckpt = load_ckpt(paths["last"], map_location=device)
        module.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        start_epoch = int(ckpt.get("epoch", 0)) + 1
        global_step = int(ckpt.get("step", 0))
        restore_rng_state(ckpt.get("rng"))
        history = list(ckpt.get("viz_history") or [])
        epoch_rows = list(ckpt.get("epoch_rows") or [])
        last_val_loss = ckpt.get("val_loss")
        print(f"Resume at epoch {start_epoch} (saved epoch={ckpt.get('epoch')}, "
              f"step={global_step}, val_loss={last_val_loss})", flush=True)

    writer = None
    tb_dir = config.get("tb_dir")
    if config.get("tensorboard") and tb_dir:
        try:
            from torch.utils.tensorboard import SummaryWriter
            writer = SummaryWriter(tb_dir)
            print(f"TensorBoard logging to {tb_dir}  (start: tensorboard --logdir=runs)", flush=True)
        except Exception as err:
            print("TensorBoard unavailable:", err, flush=True)

    accum = int(config["accum_iter"])
    grad_clip = float(config.get("grad_clip", 0.0))
    num_epochs = int(config["num_epochs"])
    steps_per_epoch = max(1, len(train_loader) // max(1, accum))
    total_steps = max(1, steps_per_epoch * num_epochs)
    warmup = int(config["warmup"])
    max_lr = float(config["base_lr"])

    def log_params(tag, step):
        if writer is None:
            return
        try:
            for name, p in module.named_parameters():
                writer.add_histogram(f"{tag}/{name}", p.detach().cpu(), step)
                if p.grad is not None:
                    writer.add_histogram(f"{tag}/grad/{name}", p.grad.detach().cpu(), step)
        except Exception:
            pass

    for epoch in range(start_epoch, num_epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss, seen, t0 = 0.0, 0, time.time()
        for i, (xb, yb) in enumerate(train_loader):
            xb, yb = xb.to(device), yb.to(device)
            _, loss = model(xb, yb)
            (loss / accum).backward()

            if (i + 1) % accum == 0:
                if grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                lr = gptv1.cosine_lr(global_step, max_lr, warmup, total_steps)
                for pg in optimizer.param_groups:
                    pg["lr"] = lr
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
                history.append({"step": global_step, "loss": total_loss / max(1, seen), "lr": lr})
                if writer is not None:
                    writer.add_scalar("train/loss", history[-1]["loss"], global_step)
                    writer.add_scalar("train/lr", lr, global_step)
                if global_step % max(500, save_every // 4) == 0:
                    log_params("step", global_step)
                if save_every > 0 and global_step % save_every == 0:
                    val_mid = gptv1.evaluate_loss(model, valid_loader, device)
                    payload = build_checkpoint(module, optimizer, epoch, global_step,
                                               config, val_mid, history, epoch_rows)
                    save_checkpoint(paths["step"](global_step), payload)
                    save_checkpoint(paths["last"], payload)
                    cleanup_old_checkpoints(config["file_prefix"], keep=3)

            total_loss += float(loss.item())
            seen += 1
            if i % 80 == 1:
                elapsed = max(1e-6, time.time() - t0)
                lr_now = optimizer.param_groups[0]["lr"]
                print(f"Epoch {epoch} | Batch {i} | Loss {total_loss / max(1, seen):.4f} | "
                      f"Speed {xb.numel() * (i + 1) / elapsed:.0f} tok/s | LR {lr_now:.8f}",
                      flush=True)

        val_loss = gptv1.evaluate_loss(model, valid_loader, device)
        last_val_loss = float(val_loss)
        val_ppl = math.exp(min(20.0, val_loss))
        train_loss = total_loss / max(1, seen)
        print("-" * 72, flush=True)
        print(f"Epoch {epoch + 1}/{num_epochs} done | train_loss={train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | val_ppl={val_ppl:.2f}", flush=True)
        print("-" * 72, flush=True)
        epoch_rows.append({"epoch": epoch, "train_loss": train_loss, "val_loss": float(val_loss),
                           "steps": global_step})
        if writer is not None:
            writer.add_scalar("val/loss", val_loss, epoch)

        payload = build_checkpoint(module, optimizer, epoch, global_step,
                                   config, last_val_loss, history, epoch_rows)
        save_checkpoint(paths["epoch"](epoch), payload)
        save_checkpoint(paths["last"], payload)
        cleanup_old_checkpoints(config["file_prefix"], keep=3)

    final_val = gptv1.evaluate_loss(model, valid_loader, device)
    last_val_loss = float(final_val)
    payload = build_checkpoint(module, optimizer, num_epochs - 1, global_step,
                               config, last_val_loss, history, epoch_rows)
    save_checkpoint(paths["final"], payload)
    save_checkpoint(paths["last"], payload)
    if writer is not None:
        writer.add_scalar("val/final_loss", final_val, num_epochs)
        writer.close()
    print("Training complete.", flush=True)
    return module, history, epoch_rows, last_val_loss


In [11]:
# Load finished model OR resume training (never retrains once final.pt exists)
def load_trained_model(tokenizer, train_loader, valid_loader, config):
    paths = checkpoint_paths(config)

    if exists(paths["final"]):
        print(f"Loading finished model {paths['final']}", flush=True)
        VIZ_STATE["did_train"] = False
        ckpt_path = paths["final"]
    elif exists(paths["last"]):
        print(f"No final checkpoint. Continuing from {paths['last']}", flush=True)
        VIZ_STATE["did_train"] = True
        _, history, epoch_rows, last_val = train_worker(tokenizer, train_loader, valid_loader, config)
        VIZ_STATE["history"], VIZ_STATE["epoch_rows"], VIZ_STATE["val_loss"] = history, epoch_rows, last_val
        ckpt_path = paths["final"] if exists(paths["final"]) else paths["last"]
    else:
        print("No checkpoint found. Starting training from scratch...", flush=True)
        VIZ_STATE["did_train"] = True
        _, history, epoch_rows, last_val = train_worker(tokenizer, train_loader, valid_loader, config)
        VIZ_STATE["history"], VIZ_STATE["epoch_rows"], VIZ_STATE["val_loss"] = history, epoch_rows, last_val
        ckpt_path = paths["final"] if exists(paths["final"]) else paths["last"]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = gptv1.make_model(tokenizer.get_vocab_size(), config).to(device)
    ckpt = load_ckpt(ckpt_path, map_location="cpu")
    state, full = unwrap_model_state(ckpt)
    model.load_state_dict(state)
    model.eval()

    if full is not None:
        VIZ_STATE["history"] = list(full.get("viz_history") or VIZ_STATE["history"] or [])
        VIZ_STATE["epoch_rows"] = list(full.get("epoch_rows") or VIZ_STATE["epoch_rows"] or [])
        VIZ_STATE["val_loss"] = full.get("val_loss", VIZ_STATE["val_loss"])
    print(f"Model ready ({gptv1.count_parameters(model):,} params). "
          f"Checkpoints use prefix: {config['file_prefix']}", flush=True)
    return model


# Part 5: Quality — charts, weights, accuracy

Training charts (loss / LR / speed over time), parameter weight/bias inspection per
block, teacher-forced token accuracy vs chance, and the quality dashboard.


In [12]:
# Training charts (altair) — loss, lr, speed, epoch progress
import pandas as pd
import altair as alt
alt.data_transformers.disable_max_rows()   # allow charts > 5000 rows

def show_training_charts(history, epoch_rows, num_epochs):
    if not history and not epoch_rows:
        print("No training history yet.")
        return []
    charts = []
    if history:
        hdf = pd.DataFrame(history)
        if "loss" in hdf.columns:
            c1 = alt.Chart(hdf).mark_line(color="#1f77b4").encode(
                x="step:Q", y=alt.Y("loss:Q", title="loss")).properties(title="Loss per step", width=420, height=200)
            charts.append(c1)
        if "lr" in hdf.columns:
            c2 = alt.Chart(hdf).mark_line(color="#d62728").encode(
                x="step:Q", y=alt.Y("lr:Q", title="learning rate")).properties(title="LR schedule (cosine)", width=420, height=200)
            charts.append(c2)
    if epoch_rows:
        edf = pd.DataFrame(epoch_rows)
        melt = edf.melt(id_vars=["epoch"], value_vars=["train_loss", "val_loss"],
                        var_name="split", value_name="loss")
        c3 = alt.Chart(melt).mark_line(point=True).encode(
            x="epoch:Q", y=alt.Y("loss:Q", title="loss"),
            color=alt.Color("split:N", title=None)).properties(title="Train vs valid loss per epoch", width=420, height=200)
        charts.append(c3)
    return charts


In [13]:
# Weight / bias inspection dashboard (per transformer block)
def weight_dashboard(model, max_blocks=None):
    module = model.module if hasattr(model, "module") else model
    rows = []
    for i, blk in enumerate(module.blocks):
        if max_blocks is not None and i >= max_blocks:
            break
        rows.append({
            "block": i,
            "attn.W (mean|std)": f"{blk.attn.c_attn.weight.data.mean():+.4f}|{blk.attn.c_attn.weight.data.std():.4f}",
            "attn.b (norm)": f"{blk.attn.c_attn.bias.data.norm():.3f}",
            "mlp.W (mean|std)": f"{blk.mlp.c_fc.weight.data.mean():+.4f}|{blk.mlp.c_fc.weight.data.std():.4f}",
            "mlp.b (norm)": f"{blk.mlp.c_fc.bias.data.norm():.3f}",
            "ln1 (mean|std)": f"{blk.ln1.weight.data.mean():+.4f}|{blk.ln1.weight.data.std():.4f}",
            "ln2 (mean|std)": f"{blk.ln2.weight.data.mean():+.4f}|{blk.ln2.weight.data.std():.4f}",
        })
    print(f"{'block':>5} | {'attn.W m|s':>20} | {'attn.b nrm':>10} | {'mlp.W m|s':>20} | {'mlp.b nrm':>10} | {'ln1 m|s':>14} | {'ln2 m|s':>14}")
    for r in rows:
        print(f"{r['block']:>5} | {r['attn.W (mean|std)']:>20} | {r['attn.b (norm)']:>10} | "
              f"{r['mlp.W (mean|std)']:>20} | {r['mlp.b (norm)']:>10} | {r['ln1 (mean|std)']:>14} | {r['ln2 (mean|std)']:>14}")
    te = module.tok_emb.weight.data
    le = module.lm_head.weight.data.norm(dim=1).mean() if module.lm_head.weight is not None else 0.0
    print(f"tok_emb (mean|std) {te.mean():+.4f}|{te.std():.4f} | lm_head row-norm mean {le:.3f}")
    return rows


In [14]:
# Token accuracy + quality dashboard
def quality_dashboard(model, valid_loader, tokenizer, config, val_loss=None):
    pad_id = tokenizer.token_to_id("<pad>")
    acc, correct, total = token_accuracy_lm(model, valid_loader, pad_id, max_batches=40)
    chance = 1.0 / tokenizer.get_vocab_size()
    ppl = float(math.exp(min(float(val_loss), 20))) if val_loss is not None else None
    print("=" * 60)
    print("QUALITY DASHBOARD")
    print(f"  teacher-forced token accuracy : {acc:.4%}  ({correct}/{total} tokens)")
    print(f"  chance (1/vocab)              : {chance:.4%}")
    print(f"  accuracy - chance             : {acc - chance:.4%}")
    print(f"  val_loss (per-token CE)       : None" if val_loss is None
          else f"  val_loss (per-token CE)       : {float(val_loss):.4f}")
    print(f"  perplexity exp(val_loss)      : {ppl:.2f}" if ppl else "  perplexity exp(val_loss)      : None")
    print("=" * 60)
    return {"token_accuracy": acc, "chance": chance, "perplexity": ppl, "val_loss": val_loss}


# Part 6: Zero-shot generation samples

The model was trained only on next-token prediction — these replies are zero-shot
(decode with temperature 0.8, top-k 40, paper-style).


In [15]:
# Zero-shot samples (called from the pipeline after model is ready)
def show_zero_shot_samples(model, vocab, samples=None):
    samples = samples or ["hello", "how are you", "solve 23*17",
                          "explain recursion in simple words", "write a short apology message"]
    for q in samples:
        print("User:", q)
        print("GPT-v1:", gptv1.generate_reply(model, vocab, q, max_new_tokens=90))
        print("-" * 60)


# Part 7: Full Pipeline — one run, all nine steps

1. config → 2. data → 3. vocab → 4. dataloaders → 5. train/resume → 6. charts →
7. weight dashboard → 8. quality dashboard → 9. zero-shot samples.


In [16]:
def run_pipeline():
    print_step(1, 9, "Data", "DailyDialog + Alpaca + GSM8K -> User:/Assistant: texts")
    train_texts, valid_texts = gptv1.build_train_valid_texts(cfg)
    print(f"Train texts: {len(train_texts)} | Valid texts: {len(valid_texts)}")

    print_step(2, 9, "Vocabulary", "word-level, saved to " + cfg["vocab_path"])
    vocab = gptv1.build_vocab(train_texts, cfg["vocab_path"])
    print(f"Vocabulary size: {vocab.get_vocab_size()}")

    print_step(3, 9, "LM blocks / dataloaders", "context=" + str(cfg["max_seq_len"]))
    train_loader, valid_loader = gptv1.create_dataloaders_for_lm(vocab, train_texts, valid_texts, cfg)
    print(f"Train batches: {len(train_loader)} | Valid batches: {len(valid_loader)}")

    print_step(4, 9, "Train or load the model", "resumes from last.pt if present")
    model = load_trained_model(vocab, train_loader, valid_loader, cfg)

    print_step(5, 9, "Final validation", "perplexity on held-out texts")
    device = next(model.parameters()).device
    val_loss = gptv1.evaluate_loss(model, valid_loader, device)
    VIZ_STATE["val_loss"] = float(val_loss)
    print(f"Validation loss: {val_loss:.4f} | Perplexity: {math.exp(min(20.0, val_loss)):.2f}")

    print_step(6, 9, "Training charts", "loss / LR history")
    charts = show_training_charts(VIZ_STATE["history"], VIZ_STATE["epoch_rows"], cfg["num_epochs"])
    for ch in charts:
        display(ch)

    print_step(7, 9, "Weight / bias dashboard", "per-block parameter stats")
    weight_dashboard(model, max_blocks=8)

    print_step(8, 9, "Quality dashboard", "token accuracy vs chance, perplexity")
    quality_dashboard(model, valid_loader, vocab, cfg, VIZ_STATE["val_loss"])

    print_step(9, 9, "Zero-shot samples", "")
    for q in ["hello", "how are you", "solve 12*13", "write a short thank-you message"]:
        print("User:", q)
        print("GPT-v1:", gptv1.generate_reply(model, vocab, q, max_new_tokens=80))
        print("-" * 50)

    return model, vocab, cfg

# Run the full pipeline
if os.environ.get("SKIP_PIPELINE") == "1":
    print("SKIP_PIPELINE=1 -> not running")
else:
    model, vocab, cfg = run_pipeline()
    print("PIPELINE COMPLETE. Checkpoints prefix:", cfg["file_prefix"], flush=True)



STEP 1/9: Data
DailyDialog + Alpaca + GSM8K -> User:/Assistant: texts

Data sources -> train: dailydialog,alpaca,gsm8k | valid: dailydialog,gsm8k
Train texts: 135026 | Valid texts: 8069

STEP 2/9: Vocabulary
word-level, saved to gptv1_vocab_local.json

Loaded vocab gptv1_vocab_local.json size=30000
Vocabulary size: 30000

STEP 3/9: LM blocks / dataloaders
context=128

Train batches: 7882 | Valid batches: 389

STEP 4/9: Train or load the model
resumes from last.pt if present

Loading finished model gptv1_local_final.pt
Model ready (56,005,632 params). Checkpoints use prefix: gptv1_local_

STEP 5/9: Final validation
perplexity on held-out texts

Validation loss: 6.0471 | Perplexity: 422.89

STEP 6/9: Training charts
loss / LR history



alt.Chart(...)

alt.Chart(...)

alt.Chart(...)


STEP 7/9: Weight / bias dashboard
per-block parameter stats

block |           attn.W m|s | attn.b nrm |            mlp.W m|s |  mlp.b nrm |        ln1 m|s |        ln2 m|s
    0 |       -0.0000|0.0000 |      0.000 |       +0.0000|0.0000 |      0.000 | +0.4704|0.0024 | +0.4705|0.0027
    1 |       +0.0000|0.0000 |      0.000 |       +0.0000|0.0000 |      0.000 | +0.4705|0.0027 | +0.4705|0.0028
    2 |       -0.0000|0.0000 |      0.000 |       +0.0000|0.0000 |      0.000 | +0.4705|0.0028 | +0.4705|0.0028
    3 |       -0.0000|0.0000 |      0.000 |       -0.0000|0.0000 |      0.000 | +0.4705|0.0029 | +0.4705|0.0029
    4 |       -0.0000|0.0000 |      0.000 |       -0.0000|0.0000 |      0.000 | +0.4705|0.0029 | +0.4705|0.0029
    5 |       +0.0000|0.0000 |      0.000 |       +0.0000|0.0000 |      0.000 | +0.4705|0.0029 | +0.4705|0.0029
    6 |       -0.0000|0.0000 |      0.000 |       +0.0000|0.0000 |      0.000 | +0.4705|0.0030 | +0.4705|0.0029
    7 |       +0.0000|0.0000 |      0.000 

# Part 8: Architecture Summary

Compact parameter table (torchinfo) + summary line. Uses `torchinfo` and `torchsummary`.


In [17]:
from torchinfo import summary as torchinfo_summary
import torchsummary

print("=" * 72)
print("GPT-v1 architecture summary (this run)")
gptv1.print_architecture(cfg, vocab_size=vocab.get_vocab_size())
print(f"Total trainable parameters: {gptv1.count_parameters(model):,}")
print(f"  - token embeddings: {model.tok_emb.weight.numel():,}")
print(f"  - position embeddings: {model.pos_emb.weight.numel():,}")
print(f"  - blocks: {len(model.blocks)} transformer decoder blocks")
print("=" * 72)

x = torch.randint(0, vocab.get_vocab_size(), (1, min(16, cfg["max_seq_len"]))).to(next(model.parameters()).device)
try:
    table = torchinfo_summary(model, input_data=x, col_names=["input_size", "output_size", "num_params", "mult_adds"], depth=4)
    print(table)
except Exception as err:
    print("torchinfo summary skipped:", err)

try:
    torchsummary.summary(model, (cfg["max_seq_len"],), dtypes=[torch.long], device="cpu")
except Exception as err:
    print("torchsummary skipped:", err)


GPT-v1 architecture summary (this run)
GPT-v1 architecture (paper-inspired)
Paper GPT-1: L=12, d=768, heads=12, d_ff=3072, ctx=512
This run: L=8, d=512, heads=8, d_ff=2048, ctx=128
Approx trainable params: 56,005,632
Total trainable parameters: 56,005,632
  - token embeddings: 15,360,000
  - position embeddings: 65,536
  - blocks: 8 transformer decoder blocks
Layer (type:depth-idx)                        Input Shape               Output Shape              Param #                   Mult-Adds
GPTDecoderOnly                                [1, 16]                   [1, 16, 30000]            --                        --
├─Embedding: 1-1                              [1, 16]                   [1, 16, 512]              15,360,000                15,360,000
├─Embedding: 1-2                              [1, 16]                   [1, 16, 512]              65,536                    65,536
├─Dropout: 1-3                                [1, 16, 512]              [1, 16, 512]              --           

# Part 9: Deep Model Visualization (colorful graphs)

| Tool | What it shows | How |
|---|---|---|
| **torchviz** | computation graph **forward + backward** edges over one real batch | `make_dot(...)` → orange/blue boxes, saved PNG |
| **TensorBoard** | **real-time** weight/bias/gradient histograms + loss curves while training | `tensorboard --logdir=runs` |
| **torchinfo/torchsummary** | parameter tables (already Part 8) | `summary(model, ...)` |
| **netron** | browser inspector of the exported ONNX graph | `netron.start(model.onnx)` |
| hiddenlayer / matplotlib | fancier architecture diagrams / heatmaps | `pip install hiddenlayer matplotlib` |


In [18]:
# Computation graph (torchviz) — forward + backward pass in color
try:
    from torchviz import make_dot
    vx = torch.randint(0, vocab.get_vocab_size(), (1, min(16, cfg["max_seq_len"]))).to(next(model.parameters()).device)
    vlogits, _ = model(vx)
    graph = make_dot(vlogits.mean(), params=dict(model.named_parameters()), show_attrs=True, show_saved=True)
    graph.render("gptv1_computation_graph", format="png", cleanup=True)
    print("Saved gptv1_computation_graph.png — the orange node is the loss (backward root);")
    print("blue boxes are parameters (weights/biases); edges carry activations + gradients.")
except Exception as err:
    print("torchviz skipped (is the graphviz binary installed? pip/apt install graphviz):", err)


Saved gptv1_computation_graph.png — the orange node is the loss (backward root);
blue boxes are parameters (weights/biases); edges carry activations + gradients.


In [19]:
# TensorBoard quick start + netron ONNX export
print("Real-time backpropagation watch:")
print("  1. This pipeline already logged scalars + weight/gradient histograms to:", cfg.get("tb_dir"))
print("  2. In a terminal run:  tensorboard --logdir=runs")
print("  3. Open http://localhost:6006 -> HISTOGRAMS tab shows weights & biases moving over time")
print("     (stepped histograms, one colorful curve per layer per step)")

try:
    vx = torch.randint(0, vocab.get_vocab_size(), (1, min(16, cfg["max_seq_len"]))).to(next(model.parameters()).device)
    torch.onnx.export(model, vx, "gptv1_model.onnx",
                      input_names=["input_ids"], output_names=["logits"],
                      opset_version=14, do_constant_folding=True)
    print("Exported gptv1_model.onnx")
    try:
        import netron
        netron.start("gptv1_model.onnx")
    except Exception as err:
        print("netron.start skipped (needs browser/network):", err)
except Exception as err:
    print("onnx export skipped:", err)


Real-time backpropagation watch:
  1. This pipeline already logged scalars + weight/gradient histograms to: runs/gptv1_local
  2. In a terminal run:  tensorboard --logdir=runs
  3. Open http://localhost:6006 -> HISTOGRAMS tab shows weights & biases moving over time
     (stepped histograms, one colorful curve per layer per step)
onnx export skipped: No module named 'onnxscript'


# Part 10: Gradio Chat UI

One-turn zero-shot chat with the trained GPT-v1 model (Gradio 6 compatible).
Opens the UI automatically; skip it with `SKIP_CHAT=1`.


In [20]:
import os as _os
if _os.environ.get("SKIP_CHAT") == "1":
    print("Chat UI skipped (SKIP_CHAT=1).")
else:
    print("Starting Gradio chat UI ... open http://127.0.0.1:7860 in your browser.")
    gptv1.launch_gradio_chat(model, vocab)


Starting Gradio chat UI ... open http://127.0.0.1:7860 in your browser.
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
